# 02 — Theory task walkthrough

Goal: explain `meridian/model/model.py` as an orchestrator over equations, transforms, priors/spec/context, and input data.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import tempfile
from pathlib import Path
from textwrap import dedent


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not find repository root from current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
WORK_ROOT = Path(tempfile.mkdtemp(prefix='meridian_expert_demo_'))
WORKSPACE = WORK_ROOT / 'runtime'

os.environ['PYTHONPATH'] = str(REPO_ROOT / 'src')
os.environ['MERIDIAN_EXPERT_WORKSPACE'] = str(WORKSPACE)
os.environ['MERIDIAN_EXPERT_LLM_BACKEND'] = 'fake'
os.environ.setdefault('OPENAI_API_KEY', 'demo-not-used-with-fake-backend')

from meridian_expert.testing_support.repo_fixtures import build_fixture_workspace

repos = build_fixture_workspace(WORK_ROOT)
os.environ['MERIDIAN_REPO_PATH'] = str(repos['meridian'])
os.environ['MERIDIAN_AUX_REPO_PATH'] = str(repos['meridian_aux'])


def run_cli(args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    cmd = ['python', '-m', 'meridian_expert.cli', *shlex.split(args)]
    result = subprocess.run(cmd, cwd=REPO_ROOT, env=os.environ.copy(), text=True, capture_output=True)
    print('$', ' '.join(cmd))
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print('--- stderr ---')
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}: {args}')
    return result


def write_demo_task(relative_path: str, body: str) -> Path:
    path = WORK_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(body).strip() + '
', encoding='utf-8')
    return path


def read_text(path: Path, max_chars: int = 1600) -> None:
    text = path.read_text(encoding='utf-8')
    print(text[:max_chars] + ('
... (truncated)' if len(text) > max_chars else ''))


print('Repo root:', REPO_ROOT)
print('Demo workspace:', WORK_ROOT)
print('Meridian fixture:', repos['meridian'])
print('Meridian_aux fixture:', repos['meridian_aux'])


## Create a concrete theory task

In [ ]:
task_path = write_demo_task('tasks/theory_model_orchestrator/task.md', '''
Explain how `meridian/model/model.py` acts as an orchestrator over
`equations.py`, `adstock_hill.py`, `transformers.py`, `spec.py`, `context.py`,
and `meridian/data/input_data.py`.

Use an engineer-facing explanation and keep assumptions explicit.
''')
create = run_cli(f'task create {task_path}')
task_id = create.stdout.strip().splitlines()[-1]
print('task_id =', task_id)


## Inspect task brief and bundle registry

In [ ]:
run_cli(f'task show {task_id} --evidence-summary')
run_cli('bundle list')
run_cli('bundle show meridian_model_core')


## Run to gate and inspect review items

In [ ]:
run_cli(f'task run {task_id} --to-gate')
reviews = run_cli(f'review queue --task-id {task_id}')
review_items = json.loads(reviews.stdout)
print('review_ids:', [item['review_id'] for item in review_items])


## Approval flow then delivery

In [ ]:
for item in review_items:
    run_cli(f"review decide {item['review_id']} approve")
run_cli(f'task run {task_id} --through-delivery')
run_cli(f'task status {task_id}')


## Inspect final answer artifact

In [ ]:
delivery_dir = WORKSPACE / 'tasks' / task_id / 'deliveries' / 'prototype' / 'D01'
print('delivery_dir:', delivery_dir)
answer_path = delivery_dir / 'answer.prototype.md'
read_text(answer_path)
